# PRISM — RAG vs Non-RAG: Column Mean Comparison

Place the two evaluation CSV files in the Google Colab **Files** panel (`/content/`), then select each file by number. This notebook calculates shared numeric-column means, differences, displays a chart, and exports CSV files.

- Text columns are excluded from averaging.
- `difficulty` is included in the summary but excluded from the performance chart.
- This is a descriptive comparison, not a statistical significance test.


## 1. Select CSV files from `/content/`


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

BASE = Path("/content")
csv_files = sorted(p for p in BASE.glob("*.csv") if p.is_file())
if len(csv_files) < 2:
    raise FileNotFoundError("Place at least two CSV files in the Colab Files panel (/content/).")

print("Available CSV files:")
for i, path in enumerate(csv_files, start=1):
    print(f"[{i}] {path.name}")

def choose_csv(label):
    while True:
        try:
            n = int(input(f"Select {label} CSV number: "))
            if 1 <= n <= len(csv_files):
                return csv_files[n - 1]
        except ValueError:
            pass
        print("Please enter a valid number from the list.")

rag_path = choose_csv("RAG")
nonrag_path = choose_csv("Non-RAG")
if rag_path == nonrag_path:
    raise ValueError("Choose two different CSV files.")

rag = pd.read_csv(rag_path)
non_rag = pd.read_csv(nonrag_path)
print(f"RAG: {rag_path.name} ({len(rag)} rows)")
print(f"Non-RAG: {nonrag_path.name} ({len(non_rag)} rows)")


## 3. Calculate mean values and differences

In [ ]:
# Only shared numeric columns are compared; preserve the original CSV column order.
shared_columns = [c for c in rag.columns if c in non_rag.columns]
metrics = [c for c in shared_columns if pd.api.types.is_numeric_dtype(rag[c]) and pd.api.types.is_numeric_dtype(non_rag[c])]
if not metrics:
    raise ValueError('No common numeric columns were found.')

summary = pd.DataFrame({
    'Metric': metrics,
    'RAG_mean': [rag[c].mean() for c in metrics],
    'Non_RAG_mean': [non_rag[c].mean() for c in metrics],
    'RAG_n': [rag[c].count() for c in metrics],
    'Non_RAG_n': [non_rag[c].count() for c in metrics],
})
summary['Difference_RAG_minus_Non_RAG'] = summary['RAG_mean'] - summary['Non_RAG_mean']
summary['Relative_change_pct'] = summary.apply(
    lambda r: 100 * (r['RAG_mean'] / r['Non_RAG_mean'] - 1) if r['Non_RAG_mean'] != 0 else float('nan'), axis=1
)
display(summary.style.format({'RAG_mean':'{:.6f}', 'Non_RAG_mean':'{:.6f}',
                              'Difference_RAG_minus_Non_RAG':'{:+.6f}', 'Relative_change_pct':'{:+.2f}%'}))

## 4. Compare evaluation metrics visually

In [ ]:
import matplotlib.pyplot as plt

# Difficulty is a dataset characteristic, not a response-quality score.
plot_metrics = [m for m in metrics if m.lower() not in ('difficulty',)]
if plot_metrics:
    plot_df = summary.set_index('Metric').loc[plot_metrics, ['RAG_mean','Non_RAG_mean']]
    ax = plot_df.rename(columns={'RAG_mean':'RAG', 'Non_RAG_mean':'Non-RAG'}).plot(kind='bar', figsize=(9,5), rot=0)
    ax.set_title('PRISM: RAG vs Non-RAG (mean scores)')
    ax.set_ylabel('Mean score')
    ax.set_xlabel('Metric')
    ax.legend()
    plt.tight_layout()
    plt.savefig('PRISM_RAG_vs_NonRAG_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('No performance metrics to plot.')

## 5. Export comparison CSVs

In [ ]:
summary.to_csv('PRISM_RAG_vs_NonRAG_column_means.csv', index=False, encoding='utf-8-sig')
summary[['Metric','RAG_mean','Non_RAG_mean','Difference_RAG_minus_Non_RAG']].to_csv(
    'PRISM_RAG_vs_NonRAG_compact_comparison.csv', index=False, encoding='utf-8-sig')
print('Saved PRISM_RAG_vs_NonRAG_column_means.csv')
print('Saved PRISM_RAG_vs_NonRAG_compact_comparison.csv')
print('Saved PRISM_RAG_vs_NonRAG_comparison.png (if plotted)')

# Optional Colab downloads:
# from google.colab import files
# files.download('PRISM_RAG_vs_NonRAG_column_means.csv')

In [ ]:

from google.colab import files

files.download("/content/PRISM_RAG_vs_NonRAG_column_means.csv")
